# Previsão de Inadimplência em Cobranças

**Case Técnico — Processo Seletivo Programa de Aceleração em Ciência de Dados**
**Datarisk**

Julho/2026

---


## Objetivo

Este documento tem como objetivo apresentar o desenvolvimento de um modelo preditivo capaz de estimar a probabilidade de inadimplência de cobranças mensais realizadas a clientes, com base no histórico de comportamento de pagamento e nas características cadastrais e de perfil disponíveis.

Considera-se inadimplente todo pagamento realizado com **5 dias ou mais de atraso em relação à data de vencimento**. As previsões finais devem ser geradas para os registros da base `base_pagamentos_teste.csv`, contendo exclusivamente a probabilidade estimada de inadimplência (valores contínuos entre 0 e 1), sem classificação binária.


## Plano de Trabalho

O desenvolvimento da solução foi estruturado nas seguintes etapas:

**Etapa 1 - Análise Exploratória de Dados (EDA)**
Investigação individual de cada uma das quatro bases (cadastral, info, pagamentos_desenvolvimento e pagamentos_teste): estrutura, tipos, nulos, duplicatas, inconsistências lógicas e distribuições relevantes. Inclui também a construção e validação da variável target (inadimplência = atraso ≥ 5 dias), feita a partir da base de desenvolvimento.

**Etapa 2 - Merges e Feature Engineering**
União das quatro bases via `ID_CLIENTE` e `SAFRA_REF`, construindo a base consolidada de modelagem. Nesta etapa também são criadas novas variáveis, tratados componentes existentes (datas, categóricas, valores faltantes) e definidas estratégias para casos especiais (clientes sem histórico, inconsistências identificadas na EDA).

**Etapa 3 - Modelagem**
Treinamento e validação de modelo(s) de machine learning para estimar a probabilidade de inadimplência, com avaliação de métricas apropriadas ao problema.

**Etapa 4 - Interpretação dos Resultados e otimização de hiperparâmetros**
Avaliação da performance do modelo, análise das variáveis mais relevantes, otimização de hiperparâmetros e validação das previsões sob a ótica de negócio, seguida da geração do arquivo final `submissao_case.csv`.

Cada etapa é documentada a seguir, com as principais decisões técnicas e suas justificativas.

---


## Etapa 1 — Análise Exploratória de Dados

### 4.1. Base Cadastral (`base_cadastral.csv`)

Esta base reúne as informações cadastrais dos clientes, com granularidade de um registro por cliente (1.315 clientes únicos).

**O que foi feito:**
- Padronização de tipos e tratamento de valores nulos.
- Teste de hipóteses para verificar se a ausência de dados em algumas colunas tinha relação com o tipo de cliente (pessoa física vs jurídica).
- Verificação de duplicatas.

**Decisões tomadas:**
- A hipótese se confirmou para `SEGMENTO_INDUSTRIAL`: 100% dos nulos dessa coluna correspondem exatamente aos clientes PF, contra apenas 1,36% de nulos entre os PJ. Isso indica que a ausência não é um erro de preenchimento, mas sim estrutural (pessoa física não possui segmento industrial). Diante disso, os nulos de PF foram recodificados como uma categoria própria, `"PESSOA_FISICA"`, preservando essa distinção, enquanto os poucos nulos remanescentes em PJ foram tratados como `"Desconhecido"`.
- A mesma hipótese foi testada para `PORTE`, mas não se confirmou: a proporção de nulos é praticamente idêntica entre PF (3,03%) e PJ (3,12%), sugerindo que a ausência é aleatória e sem relação estrutural com o tipo de cliente. Nesse caso, todos os nulos foram tratados como `"Desconhecido"`.
- Para `DDD` e `DOMINIO_EMAIL`, sem hipótese estrutural aplicável, os nulos também foram tratados como `"Desconhecido"`.
- Não foram encontradas duplicatas de linha nem de `ID_CLIENTE`, confirmando que a granularidade da base está correta (um registro único por cliente).

### 4.2. Base de Informações Mensais (`base_info.csv`)

Esta base traz dados mensais de acompanhamento dos clientes, como renda do mês anterior e número de funcionários, com granularidade de um registro por cliente por safra (24.401 registros no total).

**O que foi feito:**
- Padronização de tipos e verificação da granularidade da base.
- Análise estatística das variáveis numéricas e investigação de valores potencialmente atípicos.

**Decisões tomadas:**
- Não foram encontradas duplicatas na combinação `ID_CLIENTE` + `SAFRA_REF`, confirmando a granularidade esperada da base.
- `RENDA_MES_ANTERIOR` apresenta 717 valores nulos (~3% da base) e uma distribuição assimétrica à direita (média de 288.751 bem acima da mediana de 240.998), o que é típico de variáveis de renda/faturamento. Essa assimetria foi registrada como um ponto de atenção para a etapa de feature engineering, onde uma transformação logarítmica poderá ser avaliada.
- Os 15 registros com `RENDA_MES_ANTERIOR` abaixo de 1.000 foram avaliados e considerados pouco expressivos (menos de 0,1% da base), sendo mantidos sem tratamento especial, apenas documentados como investigados.
- `NO_FUNCIONARIOS` apresenta 1.252 valores nulos (~5% da base) e uma distribuição mais simétrica (média de 117,8 próxima da mediana de 118).
- Os 214 registros com `NO_FUNCIONARIOS == 0` foram identificados como um ponto a ser melhor compreendido antes de qualquer tratamento definitivo, com um cruzamento iniciado contra a `FLAG_PF` da base cadastral para checar se a concentração ocorre entre clientes PF (o que validaria o valor como legítimo) ou entre PJ (o que sugeriria tratar como nulo disfarçado). **Esta investigação ficou pendente de conclusão** e será retomada antes da definição final do tratamento de nulos desta base.

### 4.3. Base de Pagamentos — Desenvolvimento (`base_pagamentos_desenvolvimento.csv`)

Esta base contém o histórico de cobranças já pagas, sendo a fonte para construção da variável target e para o desenvolvimento das features comportamentais do modelo.

**O que foi feito:**
- Cálculo do atraso entre pagamento e vencimento e construção da variável target (inadimplência = atraso ≥ 5 dias).
- Análise do balanceamento da target.
- Identificação de um pequeno grupo de registros com inconsistência lógica entre datas.
- Teste de hipótese relacionando essa inconsistência à ocorrência de inadimplência.

**Decisões tomadas:**
- Inicialmente, os 27 registros com inconsistência lógica de datas foram removidos da base, sob a hipótese de que se tratava de ruído de digitação. Contudo, ao investigar a distribuição da target apenas nesses registros, constatou-se que 26 dos 27 (96,3%) eram inadimplentes — uma proporção muito acima da média geral (7%). Diante desse achado, a decisão foi revertida: os registros foram reincorporados à base, e optou-se por criar, na etapa de feature engineering, uma variável binária (`FLAG_VENCIMENTO_INCONSISTENTE`) para capturar esse sinal, em vez de descartar informação potencialmente valiosa.
- Os nulos em `VALOR_A_PAGAR` não foram tratados nesta etapa. Observou-se que, entre os registros com valor ausente, a taxa de inadimplência é de 9,4% (contra 7% na base geral), sugerindo que a ausência do dado pode carregar sinal preditivo.

### 4.4. Base de Pagamentos — Teste (`base_pagamentos_teste.csv`)

Esta base contém as cobranças mais recentes, para as quais o modelo final deve gerar as probabilidades de inadimplência.

**O que foi feito:**
- Validação de estrutura e comparação com a base de desenvolvimento.
- Checagem de sobreposição temporal entre treino e teste.
- Verificação da mesma inconsistência lógica de datas e de nulos identificados na base de desenvolvimento.
- Análise da sobreposição de clientes entre as duas bases.

**Decisões tomadas:**
- Confirmada a ausência de sobreposição temporal entre treino e teste, validando a estrutura de validação temporal do problema (treinar no passado, prever o futuro).
- A presença do mesmo padrão de inconsistência de datas na base de teste reforçou a decisão de tratar esse padrão como uma feature (`FLAG_VENCIMENTO_INCONSISTENTE`) em vez de removê-lo, garantindo que a mesma lógica de tratamento seja aplicada de forma idêntica em treino e teste.
- A existência de 88 clientes sem histórico prévio (cold start) foi registrada como um ponto de atenção para a etapa de feature engineering, exigindo uma estratégia de fallback (por exemplo, imputação de features comportamentais com base em médias de segmento/perfil, já que não há histórico individual disponível para esses casos).

## Etapa 2: Engenharia de Features

Nesta etapa, o foco foi enriquecer a base de modelagem com variáveis derivadas capazes de capturar padrões comportamentais, temporais e históricos relevantes para a previsão de inadimplência, além de tratar inconsistências identificadas na etapa de exploração e resolver o desafio de clientes presentes apenas na base de teste, sem cadastro ou histórico prévio.

- **Sequência de merges**: para ambas as bases (desenvolvimento e teste), partiu-se da base de pagamentos como referência, unida via `left join` com `base_info` pelas chaves `ID_CLIENTE` e `SAFRA_REF` (informações mensais), seguida de `left join` com `base_cadastral` pela chave `ID_CLIENTE` (informações estáticas do cliente). O uso de `how='left'` garante a preservação de todas as cobranças originais, evitando perda de registros.

- **Validação de integridade dos merges**: confirmada preservação da contagem original de linhas após a junção das quatro bases (77.414 registros em desenvolvimento e 12.275 em teste), sem duplicações por chaves repetidas.

- **Tratamento de nulos**: `VALOR_A_PAGAR`, `RENDA_MES_ANTERIOR` e `NO_FUNCIONARIOS` mantidos sem imputação, aproveitando a capacidade nativa dos algoritmos baseados em árvore de decisões de lidar com ausências sem viés artificial. Variáveis categóricas da base cadastral (segmento, porte, domínio de e-mail, DDD, CEP) preenchidas com "Desconhecido" para os clientes sem cadastro.

- **`NUMERO_COBRANCA_CLIENTE`**: variável ordinal que numera sequencialmente as cobranças de cada cliente, calculada sobre a concatenação de desenvolvimento e teste, garantindo que o histórico de desenvolvimento seja corretamente herdado pelos clientes também presentes no teste.

- **Features temporais**: extração de `DIA_COBRANCA` e `MES_COBRANCA` a partir de `DATA_VENCIMENTO`, para capturar possível sazonalidade na inadimplência.

- **`TEMPO_RELACIONAMENTO_DIAS`**: tempo em dias entre `DATA_CADASTRO` e a `SAFRA_REF` da cobrança.

- **`FLAG_REGISTRO_INCONSISTENTE`**: sinaliza casos em que `DATA_EMISSAO_DOCUMENTO` é posterior a `DATA_VENCIMENTO`, inconsistência rara identificada na exploração (29 casos).

- **`TAXA_HISTORICA_INADIMPLENCIA`**: taxa expansiva de inadimplência de cada cliente, calculada exclusivamente com base em cobranças anteriores à cobrança atual (via `cumcount` e `shift(1)` + `cumsum` sobre o target, ordenados por `ID_CLIENTE` e `DATA_VENCIMENTO`), evitando vazamento temporal. Clientes na primeira cobrança (sem histórico) resultam em `NaN`, mantido nativo e não imputado, diferenciando um histórico de inadimplência nulo e um ausente.

- **`COMPROMETIMENTO_RENDA`**: variável de razão entre `VALOR_A_PAGAR` e `RENDA_MES_ANTERIOR`, buscando capturar o peso proporcional da cobrança na renda do cliente. Valores divididos por zero foram inputados como `NaN`.

- **Validação final**: transformações checadas por contagem de linhas, amostragem de registros e inspeção de casos extremos, confirmando que as novas variáveis refletem padrões reais de comportamento, e não artefatos de processamento.

## Etapa 3 - Modelagem

#### Estratégia de Split

- Divisão treino/validação realizada de forma temporal, respeitando a ordem cronológica das safras (`SAFRA_REF`), e não de forma aleatória. Corte realizado para que o período de validação inicie em janeiro/21.
- Motivação: evitar vazamento de informação entre treino e validação, já que um mesmo cliente aparece em múltiplas safras ao longo do tempo; um split aleatório colocaria cobranças do mesmo cliente em treino e validação simultaneamente, inflando artificialmente as métricas.

#### Escolha dos Modelos e Parâmetros Iniciais

- Testamos três algoritmos de gradient boosting sobre árvores, mesma família conceitual: **LightGBM**, **XGBoost** e **CatBoost**, pois:
  - Lidam nativamente com mistura de variáveis numéricas e categóricas de alta cardinalidade, sem exigir one-hot encoding extenso.
  - Suportam ponderação de classes (`scale_pos_weight`) nativamente, adequado ao desbalanceamento da base (~7% de inadimplência), sem necessidade de reamostragem.
  - Suportam early stopping baseado em conjunto de validação, controlando overfitting de forma automática.

Sobre os modelos utilizados:

  - **XGBoost** (Chen & Guestrin, 2016): boosting regularizado (L1/L2), referência consolidada em problemas tabulares, com bom equilíbrio entre performance e velocidade de treino.
  - **LightGBM** (Ke et al., 2017): crescimento *leaf-wise* (por folha, em vez de por nível), o que o torna mais rápido em bases grandes, à custa de maior propensão a overfitting em bases menores ou mais ruidosas.
  - **CatBoost** (Prokhorenkova et al., 2018): tratamento nativo de categóricas via *ordered target statistics* (reduz vazamento durante o treino) e árvores simétricas, com maior potencial de generalização em bases com muitas categóricas de alta cardinalidade, cenário do nosso projeto.

- Resultado empírico: CatBoost superior em AUC-ROC e AUC-PR em todas as rodadas realizadas; LightGBM consistentemente inferior, provavelmente por lidar de forma menos sofisticada com variáveis categóricas de alta cardinalidade. Diante disso, **LightGBM foi descartado**, e apenas **CatBoost e XGBoost** seguiram para a etapa de otimização de hiperparâmetros.

- Parâmetros iniciais:
  - `scale_pos_weight`: razão entre negativos e positivos no conjunto de treino, seguindo recomendação padrão das bibliotecas para lidar com desbalanceamento de classes.
  - `n_estimators=500` combinado com `early_stopping_rounds=50`, deixando o número efetivo de árvores ser definido pelo ponto ótimo de validação, em vez de uma escolha arbitrária fixa.
 - `eval_metric=`AUC-PR, alinhando o critério de early stopping com a métrica de decisão final. O AUC-PR (área sob a curva de precisão x recall) é mais adequado ao desbalanceamento de classes do problema, pois é sensível ao desempenho do modelo especificamente sobre a classe minoritária (inadimplência).
  - Demais hiperparâmetros (profundidade, learning rate etc.) mantidos nos valores padrão de cada biblioteca nessa fase, para isolar o efeito do algoritmo em si antes de qualquer otimização fina.
  - Tempo de treinamento registrado para cada modelo em célula separada, permitindo comparação de custo computacional além das métricas de performance.

## Etapa 4 — Interpretação dos Resultados e Otimização de Hiperparâmetros

#### Seleção dos Modelos para Otimização

- Com o LightGBM já descartado na Etapa 3, a otimização de hiperparâmetros foi conduzida apenas sobre XGBoost e CatBoost, os dois modelos com melhor desempenho na comparação inicial.

#### Ajustes na Base Antes do Tuning

- **Associação entre `CEP_2_DIG` e `DDD`**: calculado V de Cramér (0,63), indicando associação forte* entre as duas variáveis geográficas. Combinado com a baixa importância de ambas ao modelo, decidiu-se manter apenas `CEP_2_DIG` e remover `DDD` do pipeline, reduzindo redundância.
*https://www.ibm.com/docs/pt-br/cognos-analytics/11.2.x?topic=terms-cramrs-v

- **`FLAG_PF`**: apresentou importância nula em ambos os modelos, consistente com a alta predominância de pessoa jurídica. Decidida sua remoção do pipeline final.

- **`COMPROMETIMENTO_RENDA`** Após testes considerando a variável nos modelos, identificamos piora do AUC-PR, provavelmente pelo fato de que os modelos baseados em árvore já capturam esse tipo de interação nativamente através de splits sucessivos nas duas variáveis originais. Desta forma, ela também foi removida da versão final do modelo.

#### Framework de Otimização

- A busca de hiperparâmetros foi conduzida com Optuna, utilizando o algoritmo TPE (Tree-structured Parzen Estimator) para busca bayesiana*. A escolha se deu por sua eficiência em encontrar boas soluções com menos avaliações do que grid ou random search, relevante dado o custo computacional do CatBoost. *Akiba, T. et al. (2019): Optuna: A Next-generation Hyperparameter Optimization Framework

#### Espaço de Busca — XGBoost (50 trials)

- `max_depth` (3 a 10): controla a profundidade máxima das árvores
- `learning_rate` (0,01 a 0,3, escala log): impacto multiplicativo na convergência do modelo
- `subsample` e `colsample_bytree` (0,5 a 1,0): frações de linhas e colunas amostradas por árvore, reduzindo correlação entre árvores e overfitting.
- `min_child_weight` (1 a 20): peso mínimo de instâncias para permitir uma nova partição; valores altos tornam o modelo mais conservador.
- `reg_alpha` e `reg_lambda` (1e-8 a 10,0, escala log): cobrem desde ausência de regularização até penalização forte, seguindo range de referência da documentação do Optuna.
- `scale_pos_weight` foi fixado (não otimizado) na razão entre classes negativas e positivas do treino, evitando um grau de liberdade extra na busca. `n_estimators` fixado em teto alto (1000), delegando o número efetivo de árvores ao `early_stopping_rounds=50`.

#### Espaço de Busca — CatBoost (30 trials)

- Mesmos princípios do XGBoost, com nomenclatura equivalente: `depth` (3 a 10), `learning_rate` (0,01 a 0,3, escala log), `subsample` e `colsample_bylevel` (0,5 a 1,0), `min_data_in_leaf` (1 a 50, equivalente ao `min_child_weight`), `l2_leaf_reg` (1e-2 a 10,0, escala log, única regularização disponível na biblioteca).
- Número de trials menor que o XGBoost dado o custo computacional bem mais alto por treino do CatBoost.
- Fixamos `bootstrap_type='Bernoulli'`, necessário para que o parâmetro `subsample` seja válido.

#### Resultados

- O tuning trouxe ganhos bastante distintos entre os dois modelos e inverteu a ordem de desempenho observada na comparação inicial, onde o **XGBoost** saiu de um AUC-PR de 0,50 (configuração inicial) para 0,59 no melhor trial do Optuna (+9,0 p.p.). O modelo final, treinado com os melhores parâmetros e sujeito a early stopping, interrompeu o treino na iteração 267, com AUC-PR de 0,5867 na validação.
- O **CatBoost** teve ganho mais modesto, de 0,56 para 0,57 (+1,0 p.p.), sugerindo que sua configuração padrão já estava relativamente bem calibrada para o problema, possivelmente em função de mecanismos de regularização mais sofisticados embutidos na biblioteca.

#### Decisão do Modelo Final

- Considerando o desempenho superior em AUC-PR após o tuning e o tempo de treino sensivelmente menor (relevante dado que a solução deve rodar periodicamente com atualização mensal de dados, conforme escopo do projeto), o **XGBoost foi escolhido como modelo final da solução**, sendo o modelo utilizado para gerar as previsões da base de submissão.

#### Interpretação dos resultados

- Com base no modelo treinado, foi possível identificar que a melhor feature para treinamento do modelo foi a taxa histórica de inadimplência, criada na etapa de feature engineering. 
- Ficou claro pela análise de importância que componentes regionais afetam diretamente a probabilidade de inadimplência (CEP 2 dig foi a segunda feature mais importante), como também temporais (no caso do mês de vencimento da cobrança).
- Já outras variáveis que instintivamente poderiam ser as apontadas como mais importantes provaram o contrário, como é o caso da taxa de juros e a renda anterior do cliente.
- A curva precision-recall mostra graficamente o que o valor de AUC PR próximo de 0,6 já havia introduzido: o modelo é capaz de detectar prováveis inadimplências de forma assertiva.
- Utilizando uma métrica similar, a partir de um limir = 0,5, temos um recall de 0,8372 e accuracy de 0.8219 para o nosso conjunto de validação. Este valor pode ser ajustado ao deslocar o limiar, onde é possível otimizar o valor de recall ao penalizar a accuracy do modelo*.
*https://garuda.kemdiktisaintek.go.id/documents/detail/6012858

#### Aplicação do modelo no conjunto de teste

- Modelo XGBoost otimizado aplicado ao conjunto de teste, onde foi possível obtermos uma distribuição normal das previsões com centro em 24% de chance de inadimplência e em que 75% dos valores estão abaixo de 30%.
- Considerando a taxa de inadimplencia do nosso conjunto de desenvolvimento (em torno de 7%), clientes com probabilidade acima de 0.52 estão entre os 7% de maior probabilidade.

